<img src="https://github.com/Camgur/Camgur/raw/main/chemai.png" width="300" alt="Chem AI Logo">

# Guest Lecture: **Cameron Gurwell**
Member of the Vargas-Hernandez group, M. Sc. Student

 [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChemAI-Lab/AI4Chem/blob/main/website/modules/06-atomic_simulation_environment.ipynb)


## Libraries

*   `ase` (**Atomic Simulation Environment**): A Python library for setting up, manipulating, and visualizing atomic structures.
*   `orb-models`: ORB force fields (GNN quantum chemistry calculators used for structure calculations).
*   `nglview`: Interactive 3D visualization of molecular structures within the Jupyter/Colab environment.

In [ ]:
!pip install ase
!pip install orb-models
# !pip install nglview==3.0.3
!pip install nglview==3.0.5

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

## ASE and ORB-v3

### What is ASE (Atomic Simulation Environment)?

ASE (Atomic Simulation Environment) is an open-source Python library designed to simplify atomistic simulations. It provides a user-friendly interface for setting up, manipulating, and visualizing atomic structures, and interfaces with various simulation codes and calculators. With ASE, you can:

*   **Build and modify atomic structures:** Create molecules, crystals, surfaces, and more.
*   **Define simulation parameters:** Set up calculations for energy minimization, molecular dynamics, and other tasks.
*   **Analyze results:** Extract and visualize data from simulations.
*   **Interface with different codes:** Connect to a wide range of electronic structure codes (e.g., DFT codes) and classical force fields. Some examples include LAMMPS, ORCA, Quantum Espresso, VASP, and CASTEP.

Link to the paper: https://iopscience.iop.org/article/10.1088/1361-648X/aa680e

Link to the documentation: https://ase-lib.org/

<img src="https://github.com/Camgur/Camgur/raw/main/ase_calculators.png" width="400" alt="Ase Calculators">
<br></br>
<img src="https://github.com/Camgur/Camgur/raw/main/Orbital-Materials-Logo.jpg" width="250" alt="ORB Logo">

### ORB-v3 as the Calculator

ORB-v3 is a fast Graph Neural Netwok (GNN) quantum chemistry calculator. In the context of atomistic simulations, a 'calculator' is a component that provides energies, forces, and other properties for a given atomic configuration. ORB-v3's purpose is to efficiently perform structure calculations for the atomic systems defined and managed by ASE (meaning it interfaces with ASE).

This model is a GNN which has been trained on large datasets of AIMD (*Ab Initio* Molecular Dynamics) simulations to approximate quantum calculations, making it significantly faster than other methods (like DFT) while still providing a reasonable level of accuracy for many applications. This speed is crucial for exploring larger systems or longer simulation times.

<img src="https://github.com/Camgur/Camgur/raw/main/ion_interactions.png" width="600" alt="Ion Interaction Levels">
<br></br>
<img src="https://github.com/Camgur/Camgur/raw/main/gnn_feature.png" width="400" alt="GNN Features">
<br></br>
<img src="https://github.com/Camgur/Camgur/raw/main/predict.png" width="600" alt="GNN Features">

### Predefined Structures

ASE provides a convenient way to access predefined atomic structures through `ase.collections`. `g2`, for example, contains a variety of common molecules (including H2O).

In [ ]:
# List of generic structures to be imported
from ase.collections import g2
from pprint import pprint
pprint(g2.names)

### Building a Water Molecule

Now, we will proceed to create a water molecule and set up the ORB-v3 calculator for it:

* **Creating the Molecule**: We use `ase.build.molecule('H2O')` to construct a water (H2O) molecule. This function from ASE's `build` module simplifies the creation of common molecular structures. This pulls data from the collections.

* **Loading ORB**: The `orb_v3_conservative_20_mpa(device='cpu')` function loads a pre-trained ORB-v3 model. This model will provide energies and forces. We then wrap this model with `ORBCalculator(orbff, device='cpu')` and assign it to our `atoms` object using `atoms.calc = calculator`.

* **Displaying Molecule Details**: `pprint(atoms)` is used to print a detailed representation of the `atoms` object (which is how ASE stores atomic information). `view(atoms, viewer='ngl')` allows us to view the molecular structure. X3D is specified here, but it doesn't work well with multiple atoms objects, so we will also use NGLView. It is also possible to load atoms objects into programs like **Blender** using the Beautiful Atoms package.

In [ ]:
from ase.build import molecule
from ase.visualize import view
from orb_models.forcefield import pretrained
from orb_models.forcefield.calculator import ORBCalculator

atoms = molecule('H2O')

orbff = pretrained.orb_v3_conservative_20_mpa(device='cpu')
calculator = ORBCalculator(orbff, device='cpu')
atoms.calc = calculator

print("Successfully created a water molecule and assigned the ORB calculator:")
pprint(atoms)
view(atoms, viewer='x3d')

### Vibrational Analysis

Vibrational analysis is a crucial technique in materials science and chemistry for understanding how atoms in a molecule or material move relative to each other. This motion, known as vibration, is quantized and gives rise to specific vibrational modes, each with a characteristic frequency and shape.

* In this section, we perform vibrational analysis on the water molecule using `ase.vibrations.Vibrations`. This module helps calculate and analyze the normal modes of vibration.

`vib.write_mode(n=None, kT=0.025852, nimages=60)` exports the vibrational modes. Specifically, `n=None` indicates that all modes are written. The `kT` parameter sets the temperature for calculating amplitudes, and `nimages` specifies how many frames to generate for each mode.

In [ ]:
from ase.vibrations import Vibrations

vib = Vibrations(atoms)
vib.run()
vib.summary()
vib.write_mode(n=None, kT=0.025852, nimages=60)

In [ ]:
from ase.io.animation import write_gif
from ase.io import Trajectory
vibration6 = Trajectory('/content/vib.6.traj')
write_gif('vib.6.mp4', vibration6, interval=33, rotation=('270x,90y,0z'))

In [ ]:
from IPython.display import Video
Video('/content/vib.6.mp4', embed=True) # Vibrational mode 6 for H2O

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
view(vibration6, viewer='ngl')

### Building a Slab

* **Periodic Structures**: ASE excels at working with periodic structures (crystals) which have defined periodic boundary conditions (PBC). We will create a slab of copper to analyze adsorption to a surface.

* **Building the Copper Slab**: We'll create a copper (Cu) slab with a face-centered cubic (FCC) surface using `ase.build.fcc111`. The `periodic=True` argument explicitly sets up periodic boundary conditions, essential for simulating extended surfaces.


In [ ]:
from ase.build import fcc111
from ase.build import surface

slab = fcc111('Cu', size=(4, 4, 2), vacuum=10.0, periodic=True)
slab.calc = calculator

print('Energy: ', slab.get_potential_energy())

pprint(slab)
view(slab, viewer='x3d')

In [ ]:
from ase.optimize import BFGS

# Geometry optimization in ASE
slab.rattle(0.5)
print("Rattled Energy:", atoms.get_potential_energy())

dyn = BFGS(slab, trajectory='cu_slab.traj')
dyn.run(fmax=0.01)
e_slab = slab.get_potential_energy()
print("Optimized Energy:", e_slab)

In [ ]:
view(slab, viewer='x3d')

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from ase.io import Trajectory

traj = Trajectory('/content/cu_slab.traj')
view(traj, viewer='ngl')

### Setting up N2 Adsorption

To model the adsorption of N2 onto the copper slab, we first need to define the N2 molecule and calculate its isolated potential energy.
* We use `ase.Atoms('2N', positions=[(0.0, 0.0, 0.0), (0.0, 0.0, 1.1)])` to create a nitrogen molecule with a bond length of 1.1 Å. The potential energy (`e_N2`) is calculated before adding it to the cell.

Next, the N2 molecule is placed on the copper slab using `ase.build.add_adsorbate`.
* `h` (height in angstroms) specifies the distance from the surface.
* `'ontop'` specifies the adsorption site.

In [ ]:
from ase.build import add_adsorbate
from ase.constraints import FixAtoms
from ase import Atoms

slab_ad = slab.copy()

# molecule = Atoms('2N', positions=[(0.0, 0.0, 0.0), (0.0, 1.1, 0.0)]) # Horizontal
molecule = Atoms('2N', positions=[(0.0, 0.0, 0.0), (0.0, 0.0, 1.1)]) # Vertical
molecule.calc = calculator
e_N2 = molecule.get_potential_energy()
# print(e_N2)
h = 1.85 # height in angstroms
add_adsorbate(slab_ad, molecule, h, 'ontop')

In [ ]:
view(slab_ad, viewer='x3d')

In [ ]:
constraint = FixAtoms(mask=[a.symbol != 'N' for a in slab_ad])
slab_ad.set_constraint(constraint)
dyn = BFGS(slab_ad, trajectory='adsorption.traj')
dyn.run(fmax=0.01)

In [ ]:
ad = Trajectory('/content/adsorption.traj')
view(ad, viewer='ngl')

In [ ]:
e_final = slab.get_potential_energy()
e_diff = e_slab + e_N2 - e_final
print(f'Theoretical Adsorption Energy: {e_diff * 1000:.0f} meV')
print('Experimental Adsorption Energy: ', 88 , ' meV;   Marmier, A., et al. Surface Science (1997). https://doi.org/10.1016/S0039-6028(97)00202-1')

## Graphite Exfoliation

To prepare for graphite exfoliation studies, we first need to build a graphite unit cell. ASE can help us create various crystal structures, including graphite. After constructing the unit cell, we will assign the ORB-v3 calculator to it to enable energy calculations.

### Graphite Exfoliation Setup

To create the graphite cell, we can use `ase.lattice.hexagonal.Graphite`. This is a convenient way to define the hexagonal lattice of carbon atoms.
* `symbol = 'C'`: Specifies carbon atoms.
* `latticeconstant={'a':2.46, 'c':6.7}`: Sets the lattice parameters. `'a'` defines the x/y plane, and `'c'` is 2x the interlayer spacing.

In [ ]:
from ase.lattice.hexagonal import Graphite

graphite = Graphite(symbol = 'C', latticeconstant={'a':2.46, 'c':6.7})

graphite_repeated = graphite.repeat((5, 5, 1))
view(graphite_repeated, viewer='x3d')

Here, we are investigating the interlayer spacing and its effect on the energy. Previous calculations for several different DFT functionals were calculated to compare against

In [ ]:
c_params = np.array([5.29, 5.82, 6.35, 6.70, 6.88, 7.41, 8.47, 10.58, 13.23, 15.88])
i_space = c_params/2

orb = []

for c in c_params:
  g = Graphite(symbol = 'C', latticeconstant={'a':2.46, 'c':c})
  g.calc = calculator
  orb.append(g.get_potential_energy()*1000) # meV

orb = np.array(orb)/4 # in meV/atom
orb += np.abs(orb[-1])

These are raw values in meV/atom from various functionals within Quantum Espresso

In [ ]:
lda = np.array([75.71, 1.91, -21.1, -23.67, -23.12, -18.44, -8.27, -0.97, -0.04, 0])
pbe = np.array([180.69, 75.99, 29.55, 15.00, 10.14, 2.43, -1.26, -0.58, -0.05, 0])
pbex = np.array([93.46, -2.84, -38.86, -46.01, -46.90, -43.49, -29.13, -10.20, -2.33, 0])
pbe0 = np.array([171.54, 66.87, 22.93, 10.04, 5.98, 0.12, -1.72, -0.49, -0.03, 0])
pbe0x = np.array([85.30, -11.05, -44.73, -50.34, -50.50, -45.38, -29.23, -9.79, -2.27, 0])

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
plt.axhline(0, color='#000000', linewidth=2)
ax.plot(i_space, orb, label='ORB-v3', marker='o', markersize=10, linewidth=2, linestyle='dashed', color='#E80715')
ax.plot(i_space, lda, label='LDA', marker='s', markersize=10, linewidth=2, color='#AF38F8')
ax.plot(i_space, pbe, label='PBE', marker='^', markersize=10, linewidth=2, color='#034DE1')
ax.plot(i_space, pbex, label='PBE-XDM', marker='^', markersize=10, linewidth=2, color='#19C2F7')
ax.plot(i_space, pbe0, label='PBE0', marker='D', markersize=10, linewidth=2, color='#FF6600')
ax.plot(i_space, pbe0x, label='PBE0-XDM', marker='D', markersize=10, linewidth=2, color='#FC207E')

ax.set_xlabel('Interlayer Spacing (Å)', fontsize=20)
ax.set_ylabel('Energy (meV/atom)', fontsize=20)
ax.set_ylim(-60, 100)
ax.set_xlim(2.64, 8)
ax.legend(loc='upper right', fontsize=16, frameon=False)
ax.set_aspect('auto', adjustable='box')  # force square aspect
ax.tick_params(labelbottom=True, labelleft=True, labelsize=16, width=2)
ax.spines[:].set_linewidth(2)

plt.tight_layout()
plt.show()

#### **Dispersion**

This is a significant result, as it shows that the graphite exfoliation cannot be accurately determined without accounting for dispersion (Van der Waals interactions). The current ORB-v3 model was not trained on dispersion data, so we will need to plug in a new model. Thankfully, ASE makes this very simple, and we can implement ORB-v2-D3, which is trained on datasets with the D3 dispersion correction.
* Note: The dispersion used in the quantum calculationsis actually XDM, which is more accurate than D3. D3 uses coordination information to determine the interaction strength.
<br>
<img src="https://github.com/Camgur/Camgur/raw/main/dispersion.png" width="400" alt="Dispersion Correction Effect">

In [ ]:
orbdisp = pretrained.orb_d3_v2(device='cpu')
dispcalc = ORBCalculator(orbdisp, device='cpu')

orbd3 = []

for c in c_params:
  g = Graphite(symbol = 'C', latticeconstant={'a':2.46, 'c':c})
  g.calc = dispcalc
  orbd3.append(g.get_potential_energy()*1000) # meV

orbd3 = np.array(orbd3)/4 # in meV/atom
orbd3 += np.abs(orbd3[-1])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 8))
plt.axhline(0, color='#000000', linewidth=2)
ax.plot(i_space, orb, label='ORB-v3', marker='o', markersize=10, linewidth=2, linestyle='dashed', color='#E80715')
ax.plot(i_space, orbd3, label='ORB-v2-D3', marker='o', markersize=10, linewidth=2, linestyle='dashed', color='#52079D')
ax.plot(i_space, lda, label='LDA', marker='s', markersize=10, linewidth=2, color='#AF38F8')
ax.plot(i_space, pbe, label='PBE', marker='^', markersize=10, linewidth=2, color='#034DE1')
ax.plot(i_space, pbex, label='PBE-XDM', marker='^', markersize=10, linewidth=2, color='#19C2F7')
ax.plot(i_space, pbe0, label='PBE0', marker='D', markersize=10, linewidth=2, color='#FF6600')
ax.plot(i_space, pbe0x, label='PBE0-XDM', marker='D', markersize=10, linewidth=2, color='#FC207E')

ax.set_xlabel('Interlayer Spacing (Å)', fontsize=20)
ax.set_ylabel('Energy (meV/atom)', fontsize=20)
ax.set_ylim(-60, 100)
ax.set_xlim(2.64, 8)
ax.legend(loc='upper right', fontsize=16, frameon=False)
ax.set_aspect('auto', adjustable='box')  # force square aspect
ax.tick_params(labelbottom=True, labelleft=True, labelsize=16, width=2)
ax.spines[:].set_linewidth(2)

plt.tight_layout()
plt.show()

### Nudged Elastic Band

#### **Setup**

Nudged Elastic Band (NEB) is a method used to find the minimum energy pathway (MEP) between two known states (initial and final states) of a system. It's particularly useful for studying diffusion processes, chemical reactions, and other transitions where an energy barrier needs to be overcome. We will use it determine the energy of activation necessary for lithium to diffuse through this solid-state conductor.

* We import `NEB` and `NEBTools` from `ase.mep` (Minimum Energy Path) module.  
  * `NEB` is used to define and run the NEB calculation.
  * `NEBTools` provides utilities for analyzing the results, such as plotting the energy barrier
  * `read` will construct atomic structures from many different file formats. We will load a **CIF** (Crystallography Information File).

* The `curl` command simply downloads the CIF for `LiAlO2` from my GitHub repo. This CIF contains the structural data for our system.

* **Initial and Final States**: After loading the CIF into an `atoms` object and assigning the `calculator` to it, we define the `initial` and `final` states for our NEB calculation. These states represent the start and end points of an atomic migration pathway.

  * `cif_path`, `idx1`, and `idx2` variables are initialized to specify the downloaded CIF file and the indices of the atoms that will be removed to define the migration event.
  * The `initial` state is created by copying the `atoms` object and deleting the atom at index `idx2`. This represents the system before the migration of the atom at `idx2`.
  * The `final` state is created by copying the `atoms` object and deleting the atom at index `idx1`. This represents the system after the atom that was originally at `idx1` has migrated to a new position (which is the original position of `idx2`).

<br>
<img src="https://github.com/Camgur/Camgur/raw/main/LiAlO2_dft.png" width="800" alt="Li Diffusion in LiAlO2">
<br>

By doing this, we simulate the movement of an atom from one site to another by considering the absence of the migrating atom at its starting and ending points, respectively, within the framework of a defect migration. This prepares the system for the NEB calculation, which will find the transition state between these two configurations. Importantly, this is technically unphysical and potentially invalid if the starting and ending configurations do not have the same energy (i.e., different coordination environments).

In [ ]:
import os
from ase.io import read
from ase.mep import NEB, NEBTools

!curl https://raw.githubusercontent.com/Camgur/Camgur/refs/heads/main/opt_LiAlO2_430184.cif -o LiAlO2.cif

In [ ]:
# Input handling
cif_path, idx1, idx2 = '/content/LiAlO2.cif', 2, 1
filename = os.path.splitext(os.path.basename(cif_path))[0]

# Establish initial/final states
atoms = read(cif_path)
atoms.calc = calculator

initial, final = atoms.copy(), atoms.copy()
del initial[idx2]
del final[idx1]

In [ ]:
view(initial, viewer='x3d')

In [ ]:
view(final, viewer='x3d')

In [ ]:
# Initial optimization of initial/final states
for state, label in zip([initial, final], ["init", "fin"]):
    state.calc = calculator
    opt = BFGS(state, trajectory=f"{filename}_{label}.traj")
    opt.run(fmax=5e-2, steps=20)

In [ ]:
# NEB init
images = [initial] + [initial.copy() for _ in range(5)] + [final]
for image in images:
    image.calc = calculator
neb = NEB(images, climb=True, allow_shared_calculator=True)
neb.interpolate()

In [ ]:
# Optimize NEB
traj_path = f"{filename}_{idx1}to{idx2}.traj"
opt = BFGS(neb, trajectory=traj_path)
opt.run(fmax=0.1, steps=20)

The `read` function may also be used with `@n:m` following the file extension to selectively read trajectory indices.

In [ ]:
traj = read(traj_path + '@-7:')
for img in traj:
    img.calc = calculator
energies = np.array([img.get_potential_energy() for img in traj])*1000 # meV

print('Energies [meV]:', energies)

energies += np.abs(energies[-1])

print(f'Relative Energies [meV]: {np.round(energies)}')

In [ ]:
neb = NEBTools(traj)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
neb.plot_band(ax=ax)

ax.set_xlabel('Coordinate (Å)', fontsize=20)
ax.set_ylabel('Energy (eV)', fontsize=20)
ax.set_aspect('auto', adjustable='box')  # force square aspect
ax.tick_params(labelbottom=True, labelleft=True, labelsize=16, width=2)
ax.spines[:].set_linewidth(2)

plt.tight_layout()
plt.show()

### Molecular Dynamics

Molecular Dynamics (MD) simulations are used to study the time-dependent behavior of atomic systems. By integrating Newton's equations of motion, MD can provide insights into material properties, phase transitions, and reaction mechanisms at finite temperatures. Here, we set up a simulation for the well-known fast ion conductor LGPS (Lithium Germanium Phosphorus Sulfide) to determine the theoretical lithium diffusion within the material.

* Note: MD simulations typically require larger unit cell known as **supercells** to avoid PBC issues, but we will use the standard unit cell for computational efficiency.
<br>
<img src="https://github.com/Camgur/Camgur/raw/main/LGPS_30161.png" width="800" alt="LGPS Diffusion Isosurface">

In [ ]:
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.logger import MDLogger
from ase.units import fs

!curl https://raw.githubusercontent.com/Camgur/Camgur/refs/heads/main/opt_LGPS.cif -o LGPS.cif

In [ ]:
cif_path, temp = '/content/LGPS.cif', 1000 # K
filename = os.path.splitext(os.path.basename(cif_path))[0]

cell = read(cif_path)
cell.calc = calculator

view(cell, viewer='x3d')

#### **Initializing MD**

Before running the Molecular Dynamics simulation, the system must be properly initialized with atomic velocities, and a thermostat must be chosen to control the temperature.

* **Maxwell-Boltzmann Velocity Distribution**: This assigns initial velocities to all atoms in the `cell` object according to the Maxwell-Boltzmann distribution at the specified `temp` (in Kelvin). This ensures that the system starts with a realistic distribution of kinetic energies.

* **Nose-Hoover Chain Thermostat**: The `NoseHooverChainNVT` class is used to set up a Molecular Dynamics simulation in the canonical ensemble (NVT, constant number of particles, volume, and temperature). This thermostat is generally more robust and accurate for maintaining a constant temperature compared to simpler thermostats like Berendsen.
  * `atoms`: The `atoms` object representing the LGPS crystal is passed to the MD simulation.
  * `temperature_K`: The target simulation temperature in Kelvin.
  * `timestep`: Defines the integration time step for the MD simulation. A large timestep was chosen here for computational efficiency, but the results should not be trusted for a production run.
  * `tdamp`: This parameter controls the coupling strength of the thermostat to the system, indicating how quickly the system's temperature should relax to the target temperature.
  * `trajectory`: Specifies the path for saving the trajectory throughout the simulation.
  * `logfile`: Specifies the path for saving thermodynamic properties at each step of the simulation.
  * `loginterval`: Determines how frequently (in simulation steps) data is written to the logfile.

In [ ]:
MaxwellBoltzmannDistribution(cell, temperature_K=temp)

md = NoseHooverChainNVT(
    atoms=cell,
    temperature_K=temp,  # in K, set by slurm
    timestep=5*fs,  # in femto-seconds
    tdamp=100*fs,    # 100 * timestep ideally
    trajectory=f'production_{filename}_{str(temp)}.traj',
    logfile=f'production_{filename}_{str(temp)}.log',
    loginterval=1,
)

In [ ]:
md.attach(MDLogger(md, cell, '-'), interval=1)
md.run(100)  # 1 ps

In [ ]:
dyn = Trajectory('/content/production_LGPS_1000.traj')
view(dyn, viewer='ngl')

In [ ]:
pprint(dyn[0][4]) # Starting Li index
pprint(dyn[0][23]) # Ending Li index

This analysis toolkit is handy for calculating MD diffusion without the heavy work of developing your own script. The calculation was short so our results are highly untrustworthy, but they predict a very fast diffusion coefficient.

In [ ]:
from ase.md.analysis import DiffusionCoefficient

# print(np.arange(4, 24, 1))
diff = DiffusionCoefficient(traj=dyn, timestep=10*fs, atom_indices=np.arange(4, 24, 1), molecule=False)
print(f'Diffusion Coefficient: {float(diff.get_diffusion_coefficients()[0][0])*(10e-8)**2 / 10e-15:.5f} cm^2/s')

In [ ]:
diff.plot()